In [0]:
!pip install pymupdf

In [0]:
file_path = "/Volumes/workspace/default/test_volume/dpact.pdf"

In [0]:
!pip install python-dotenv

In [0]:
from dotenv import load_dotenv
import os
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")



# Set environment variable
os.environ["OPENAI_API_KEY"] = openai_api_key

In [0]:
import fitz

#Read file content
with open(file_path, "rb") as f:
  pdf_bytes = f.read()
#Create PDF document
doc = fitz.open("pdf",pdf_bytes)
#Extract text from PDF document
text = ""
for page in doc:
    text += page.get_text()

print(text)


In [0]:
!pip install langchain-text-splitters

In [0]:
!pip install langchain

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import pandas as pd
raw_text = text
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)     
docs = text_splitter.create_documents([raw_text])
docs
len(docs)
pd_docs = pd.DataFrame([doc.dict() for doc in docs])
pd_docs.insert(0, "id_pk",range(1, len(pd_docs)+1))
display(pd_docs)



In [0]:
spark_df = spark.createDataFrame(pd_docs[['id_pk','page_content']])
spark_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.default.my_data_chunks")


                                                                                              
                                                                                              

In [0]:
%pip install databricks-vectorsearch


In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.vector_search.client import VectorSearchClient
client  = VectorSearchClient()

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

index = client.get_index(
    endpoint_name="vs_index",
    index_name="workspace.default.pdf_index"
)

result = index.similarity_search(
    query_text="who is data fiduciary",
    columns=["id_pk", "page_content"],
    num_results=3
)

display(result)

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

print(client.list_indexes(name="vs_index"))

In [0]:
client.list_endpoints()


Index was created using the UI

In [0]:
index = client.get_index(index_name = "workspace.default.vs_index")
index

In [0]:
chat_model =  ChatDatabricks(
    endpoint=  "databricks-meta-llama-3-3-70b-instruct",
    temperature= 0.1,
    max_tokens = 250,
)
chat_model.invoke(str(input))